In [ ]:
#### win rate analysis ####
import json

baseline_jsonl_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/generate_images/drawbench/dpo-official/ckpt-0/evaluation_results.jsonl"
pickscore_002_jsonl_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/generate_images/drawbench/pickscore_0.02/ckpt-500/evaluation_results.jsonl"

# Load JSONL file
def load_jsonl(file_path):
    data = {}
    with open(file_path, 'r') as f:
        for line in f:
            item = json.loads(line.strip())
            data[item['sample_id']] = item
    return data

baseline_data = load_jsonl(baseline_jsonl_path)
pickscore_002_data = load_jsonl(pickscore_002_jsonl_path)

print(f"Baseline samples: {len(baseline_data)}")
print(f"Pickscore_0.02 samples: {len(pickscore_002_data)}")

# Calculate win_rate
def calculate_win_rate(baseline_data, pickscore_data, score_key='pickscore'):
    wins = 0
    ties = 0
    losses = 0
    total = 0
    
    common_ids = set(baseline_data.keys()) & set(pickscore_data.keys())
    
    for sample_id in sorted(common_ids):
        baseline_score = baseline_data[sample_id]['scores'][score_key]
        pickscore_score = pickscore_data[sample_id]['scores'][score_key]
        
        if pickscore_score > baseline_score:
            wins += 1
        elif pickscore_score == baseline_score:
            ties += 1
        else:
            losses += 1
        total += 1
    
    win_rate = wins / total if total > 0 else 0
    
    return {
        'win_rate': win_rate,
        'wins': wins,
        'ties': ties,
        'losses': losses,
        'total': total
    }

# Define all metrics to calculate
metrics_list = ['pickscore', 'imagereward', 'clip_iqa', 'deqa', 'aesthetic_v2_5',]

# Get metrics that exist in both datasets
sample_id = list(baseline_data.keys())[0]
available_metrics = [m for m in metrics_list 
                     if m in baseline_data[sample_id]['scores'] 
                     and m in pickscore_002_data[sample_id]['scores']]

print(f"\nAvailable metrics: {available_metrics}")
print("\n=== Win Rate Analysis (Pickscore_0.02 vs Baseline) ===\n")

# 计算所有指标的win_rate
results = {}
for metric in available_metrics:
    win_rate_info = calculate_win_rate(baseline_data, pickscore_002_data, score_key=metric)
    results[metric] = win_rate_info
    print(f"{metric:20s}: {win_rate_info['win_rate']:6.4f} ({win_rate_info['win_rate']*100:6.2f}%) | "
          f"Wins: {win_rate_info['wins']:4d}, Ties: {win_rate_info['ties']:4d}, Losses: {win_rate_info['losses']:4d} | "
          f"Total: {win_rate_info['total']}")

# print("\n=== 汇总 ===")
# print(f"最佳指标 (win_rate最高): {max(results.items(), key=lambda x: x[1]['win_rate'])[0]}")
# print(f"最差指标 (win_rate最低): {min(results.items(), key=lambda x: x[1]['win_rate'])[0]}")


In [ ]:
#### Score Diff Analysis ####
import json
import numpy as np
import matplotlib.pyplot as plt

baseline_jsonl_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/generate_images/drawbench/dpo-official/ckpt-0/evaluation_results.jsonl"
pickscore_002_jsonl_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/generate_images/drawbench/pickscore_0.02/ckpt-500/evaluation_results.jsonl"

# Load JSONL file
def load_jsonl(file_path):
    data = {}
    with open(file_path, 'r') as f:
        for line in f:
            item = json.loads(line.strip())
            data[item['sample_id']] = item
    return data

baseline_data = load_jsonl(baseline_jsonl_path)
pickscore_002_data = load_jsonl(pickscore_002_jsonl_path)

# Get common sample_ids
common_ids = sorted(set(baseline_data.keys()) & set(pickscore_002_data.keys()))

# Calculate pickscore difference
score_key = 'pickscore'
score_diffs = []
score_details = []

for sample_id in common_ids:
    baseline_score = baseline_data[sample_id]['scores'][score_key]
    pickscore_002_score = pickscore_002_data[sample_id]['scores'][score_key]
    diff = pickscore_002_score - baseline_score
    score_diffs.append(diff)
    score_details.append({
        'sample_id': sample_id,
        'baseline_score': baseline_score,
        'pickscore_002_score': pickscore_002_score,
        'diff': diff
    })

score_diffs = np.array(score_diffs)

# Sort by difference and find top 12 samples with largest differences
score_details_sorted = sorted(score_details, key=lambda x: x['diff'], reverse=True)
top_12 = score_details_sorted[:60]

# Output top 12 samples with largest differences
print("="*80)
print("Top 12 samples with largest differences (Pickscore_0.02 - Baseline):")
print("="*80)
print(f"{'ID':<8} {'Baseline Score':<18} {'Pickscore_0.02 Score':<22} {'Difference':<12}")
print("-"*80)
for item in top_12:
    print(f"{item['sample_id']:<8} {item['baseline_score']:<18.6f} {item['pickscore_002_score']:<22.6f} {item['diff']:<12.6f}")
print()

# Plot difference distribution histogram
plt.figure(figsize=(10, 6), dpi=300)
plt.hist(score_diffs, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero difference line')
plt.axvline(np.mean(score_diffs), color='green', linestyle='--', linewidth=2, label=f'Mean: {np.mean(score_diffs):.4f}')
plt.xlabel('Pickscore Difference (Pickscore_0.02 - Baseline)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Pickscore Difference Distribution Histogram', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
